In [1]:
import arviz as az
import numpy as np
import pandas as pd

In [101]:
#https://www.probability.ca/jeff/ftpdir/confbias.pdf

In [5]:
def mcmc_summary_statistics(df,true_values,orden,tiempo,burnin):
    # Diccionario para almacenar los resultados
    summary_stats = {}
    c=0
    df = df[orden]
    df=df.iloc[burnin:,]
    for column in df.columns:
        trace = df[column].dropna()  # Eliminar valores NaN si los hay
        
        # Cálculo del Absolute Bias
        # Supongamos que el valor verdadero (true_value) es conocido
        # Si no es conocido, se puede calcular el sesgo con respecto al valor estimado de la media posterior
        true_value = true_values[c]  # Sustituir esto por el valor verdadero del parámetro si se conoce
        absolute_bias = np.abs(np.mean(trace) - true_value)
        bias = np.mean(trace) - true_value
        # Cálculo del Standard Error
        standard_error = np.std(trace)# / np.sqrt(len(trace))
        desviacion = np.std(trace) 
        # Cálculo del 95% CI (Intervalo de Confianza)
        lower_bound, upper_bound = np.percentile(trace, [2.5, 97.5]) #97.5-2.5=95
        ci_length = upper_bound - lower_bound
        
        # Almacenar los resultados
        summary_stats[column] = {
            'Absolute Bias': absolute_bias,
            'Standard Error': standard_error,
            '95% CI Length': ci_length,
            'media': np.mean(trace),
            'desviacion': desviacion,
            'lower_bound':lower_bound,
            'upper_bound':upper_bound,
            'bias':bias
        }
        c+=1
    # Convertir los resultados en un DataFrame
    summary_df = pd.DataFrame(summary_stats).T
    
    data = az.convert_to_inference_data(df[orden].to_dict('list'))
    resumen = az.summary(data)
    summary_df['ess']=resumen['ess_bulk'].values
    summary_df['mcse']=resumen['mcse_mean'].values
    summary_df['ess_min']=round(summary_df['ess']/tiempo,0)
    return summary_df

# Aplicación

In [27]:
df=pd.read_csv('Traceplot_aplicacion_phi_final/trace_covariables_D4_aplicacion_M5.csv')
orden = ['posterior_g1', 'posterior_g2', 'posterior_g3',
       'posterior_g4','posterior_g5', 'posterior_phi','posterior_sigma','posterior_beta3','posterior_rho']

d=mcmc_summary_statistics(df,[np.exp(1),1,1,1,1,0.5,1,5,0.5],orden,90,0)
for row in d.T.itertuples():
    if row[0]!='ess_min':
        print('& '.join([f"{x:.2f}" for x in row[1:]]))
    else:
        print('& '.join([f"{x:.0f}" for x in row[1:]]))



d=mcmc_summary_statistics(df,[np.exp(1),1,1,1,1,0.8,0.7,5,0.5],orden,90,7500)
display(d.T)
for row in d.T.itertuples():
    if row[0]!='ess_min':
        print('& '.join([f"{x:.2f}" for x in row[1:]]))
    else:
        print('& '.join([f"{x:.0f}" for x in row[1:]]))

arviz - WARNING - Shape validation failed: input_shape: (1, 10000), minimum_shape: (chains=2, draws=4)
arviz - WARNING - Shape validation failed: input_shape: (1, 2500), minimum_shape: (chains=2, draws=4)


1.17& 0.23& 1.14& 0.75& 0.20& 0.10& 0.87& 2.92& 1.16
1.18& 1.19& 0.10& 0.13& 1.18& 0.25& 0.04& 0.09& 0.29
4.53& 4.55& 0.40& 0.51& 4.52& 0.95& 0.16& 0.32& 1.15
1.55& 0.77& -0.14& 0.25& 0.80& 0.60& 0.13& 2.08& 1.66
1.18& 1.19& 0.10& 0.13& 1.18& 0.25& 0.04& 0.09& 0.29
-0.82& -1.41& -0.34& -0.00& -1.34& -0.07& 0.04& 2.00& 1.17
3.71& 3.14& 0.06& 0.50& 3.18& 0.88& 0.21& 2.32& 2.32
9640.00& 8858.00& 8940.00& 9469.00& 9759.00& 9098.00& 9766.00& 10441.00& 10252.00
0.00& 0.01& 0.01& 0.00& 0.00& 0.01& 0.00& 0.00& 0.00
107& 98& 99& 105& 108& 101& 109& 116& 114


,posterior_g1,posterior_g2,posterior_g3,posterior_g4,posterior_g5,posterior_phi,posterior_sigma,posterior_beta3,posterior_rho
Absolute Bias,1.181980,0.227176,1.136581,0.751893,0.183878,0.197622,0.569725,2.918139,1.166514
Standard Error,1.194837,1.202765,0.101155,0.131130,1.193408,0.248812,0.041532,0.091539,0.286104
95% CI Length,4.549872,4.646914,0.395695,0.512224,4.562818,0.963265,0.163579,0.318933,1.150267
media,1.536302,0.772824,-0.136581,0.248107,0.816122,0.602378,0.130275,2.081861,1.666514
desviacion,1.194837,1.202765,0.101155,0.131130,1.193408,0.248812,0.041532,0.091539,0.286104
lower_bound,-0.820812,-1.471690,-0.338855,-0.005064,-1.372497,-0.087054,0.041088,2.000000,1.168924
upper_bound,3.729060,3.175223,0.056840,0.507160,3.190322,0.876211,0.204667,2.318933,2.319192
ess,2220.000000,2244.000000,2236.000000,2381.000000,2364.000000,2241.000000,2272.000000,2615.000000,2374.000000
mcse,0.002000,0.025000,0.025000,0.002000,0.003000,0.025000,0.005000,0.006000,0.001000
ess_min,25.000000,25.000000,25.000000,26.000000,26.000000,25.000000,25.000000,29.000000,26.000000


1.18& 0.23& 1.14& 0.75& 0.18& 0.20& 0.57& 2.92& 1.17
1.19& 1.20& 0.10& 0.13& 1.19& 0.25& 0.04& 0.09& 0.29
4.55& 4.65& 0.40& 0.51& 4.56& 0.96& 0.16& 0.32& 1.15
1.54& 0.77& -0.14& 0.25& 0.82& 0.60& 0.13& 2.08& 1.67
1.19& 1.20& 0.10& 0.13& 1.19& 0.25& 0.04& 0.09& 0.29
-0.82& -1.47& -0.34& -0.01& -1.37& -0.09& 0.04& 2.00& 1.17
3.73& 3.18& 0.06& 0.51& 3.19& 0.88& 0.20& 2.32& 2.32
2220.00& 2244.00& 2236.00& 2381.00& 2364.00& 2241.00& 2272.00& 2615.00& 2374.00
0.00& 0.03& 0.03& 0.00& 0.00& 0.03& 0.01& 0.01& 0.00
25& 25& 25& 26& 26& 25& 25& 29& 26


# Pasquier

In [6]:
df=pd.read_csv('Traceplot_simulacion_phi/estimacion_t_simulacion_D8_extremo_V1_V11_c1.csv')
orden = ['posterior_g1', 'posterior_g2', 'posterior_g3',
       'posterior_g4', 'posterior_phi','posterior_sigma','posterior_beta3','posterior_rho']

d=mcmc_summary_statistics(df,[np.exp(1),1,1,1,0.5,1,5,0.5],orden,90,0)
for row in d.T.itertuples():
    if row[0]!='ess_min':
        print('& '.join([f"{x:.2f}" for x in row[1:]]))
    else:
        print('& '.join([f"{x:.0f}" for x in row[1:]]))



d=mcmc_summary_statistics(df,[np.exp(1),1,1,1,0.8,0.7,5,0.5],orden,90,7500)
display(d.T)
for row in d.T.itertuples():
    if row[0]!='ess_min':
        print('& '.join([f"{x:.2f}" for x in row[1:]]))
    else:
        print('& '.join([f"{x:.0f}" for x in row[1:]]))

arviz - WARNING - Shape validation failed: input_shape: (1, 10000), minimum_shape: (chains=2, draws=4)
arviz - WARNING - Shape validation failed: input_shape: (1, 2500), minimum_shape: (chains=2, draws=4)


0.12& 0.01& 0.06& 0.10& 0.04& 0.01& 0.62& 0.07
0.23& 0.10& 0.10& 0.13& 0.09& 0.07& 1.02& 0.19
0.89& 0.39& 0.38& 0.52& 0.37& 0.28& 4.09& 0.72
2.60& 1.01& 0.94& 1.10& 0.46& 1.01& 4.38& 0.57
0.23& 0.10& 0.10& 0.13& 0.09& 0.07& 1.02& 0.19
2.16& 0.82& 0.75& 0.84& 0.28& 0.83& 2.44& 0.25
3.05& 1.21& 1.13& 1.36& 0.65& 1.10& 6.53& 0.96
-0.12& 0.01& -0.06& 0.10& -0.04& 0.01& -0.62& 0.07
8661.00& 9473.00& 10074.00& 9353.00& 9545.00& 9698.00& 9168.00& 9675.00
0.01& 0.00& 0.00& 0.00& 0.00& 0.00& 0.00& 0.00
96& 105& 112& 104& 106& 108& 102& 108


,posterior_g1,posterior_g2,posterior_g3,posterior_g4,posterior_phi,posterior_sigma,posterior_beta3,posterior_rho
Absolute Bias,0.112986,0.012642,0.064562,0.100604,0.341021,0.304671,0.611619,0.070593
Standard Error,0.234635,0.100062,0.097255,0.133903,0.088713,0.065181,1.068804,0.195413
95% CI Length,0.907702,0.387240,0.379789,0.517882,0.362059,0.272835,4.264664,0.715870
media,2.605296,1.012642,0.935438,1.100604,0.458979,1.004671,4.388381,0.570593
desviacion,0.234635,0.100062,0.097255,0.133903,0.088713,0.065181,1.068804,0.195413
lower_bound,2.150865,0.816642,0.744528,0.838960,0.287984,0.825496,2.377369,0.245022
upper_bound,3.058567,1.203882,1.124317,1.356843,0.650044,1.098331,6.642033,0.960892
bias,-0.112986,0.012642,-0.064562,0.100604,-0.341021,0.304671,-0.611619,0.070593
ess,2145.000000,2348.000000,2339.000000,2334.000000,2161.000000,2330.000000,2169.000000,2381.000000
mcse,0.023000,0.005000,0.002000,0.002000,0.003000,0.002000,0.004000,0.001000


0.11& 0.01& 0.06& 0.10& 0.34& 0.30& 0.61& 0.07
0.23& 0.10& 0.10& 0.13& 0.09& 0.07& 1.07& 0.20
0.91& 0.39& 0.38& 0.52& 0.36& 0.27& 4.26& 0.72
2.61& 1.01& 0.94& 1.10& 0.46& 1.00& 4.39& 0.57
0.23& 0.10& 0.10& 0.13& 0.09& 0.07& 1.07& 0.20
2.15& 0.82& 0.74& 0.84& 0.29& 0.83& 2.38& 0.25
3.06& 1.20& 1.12& 1.36& 0.65& 1.10& 6.64& 0.96
-0.11& 0.01& -0.06& 0.10& -0.34& 0.30& -0.61& 0.07
2145.00& 2348.00& 2339.00& 2334.00& 2161.00& 2330.00& 2169.00& 2381.00
0.02& 0.01& 0.00& 0.00& 0.00& 0.00& 0.00& 0.00
24& 26& 26& 26& 24& 26& 24& 26


In [7]:
df=pd.read_csv('Traceplot_simulacion_phi/estimacion_t_simulacion_D8_extremo_V2_V12_c1.csv')
orden = ['posterior_g1', 'posterior_g2', 'posterior_g3',
       'posterior_g4', 'posterior_phi','posterior_sigma','posterior_beta3','posterior_rho']

d=mcmc_summary_statistics(df,[np.exp(1),1,1,1,0.5,1,5,0.5],orden,90,0)
for row in d.T.itertuples():
    if row[0]!='ess_min':
        print('& '.join([f"{x:.2f}" for x in row[1:]]))
    else:
        print('& '.join([f"{x:.0f}" for x in row[1:]]))



d=mcmc_summary_statistics(df,[np.exp(1),1,1,1,0.8,0.7,5,0.5],orden,90,7500)
display(d.T)
for row in d.T.itertuples():
    if row[0]!='ess_min':
        print('& '.join([f"{x:.2f}" for x in row[1:]]))
    else:
        print('& '.join([f"{x:.0f}" for x in row[1:]]))

arviz - WARNING - Shape validation failed: input_shape: (1, 10000), minimum_shape: (chains=2, draws=4)
arviz - WARNING - Shape validation failed: input_shape: (1, 2500), minimum_shape: (chains=2, draws=4)


0.10& 0.01& 0.10& 0.00& 0.08& 0.04& 0.22& 0.01
0.22& 0.09& 0.09& 0.12& 0.09& 0.07& 1.52& 0.21
0.87& 0.36& 0.37& 0.45& 0.34& 0.31& 6.19& 0.77
2.61& 0.99& 0.90& 1.00& 0.42& 1.04& 5.22& 0.51
0.22& 0.09& 0.09& 0.12& 0.09& 0.07& 1.52& 0.21
2.18& 0.81& 0.71& 0.77& 0.24& 0.83& 2.86& 0.20
3.04& 1.17& 1.08& 1.22& 0.58& 1.14& 9.05& 0.97
-0.10& -0.01& -0.10& 0.00& -0.08& 0.04& 0.22& 0.01
8684.00& 9044.00& 9890.00& 9994.00& 10234.00& 10224.00& 9863.00& 9873.00
0.02& 0.00& 0.00& 0.00& 0.00& 0.00& 0.00& 0.00
96& 100& 110& 111& 114& 114& 110& 110


,posterior_g1,posterior_g2,posterior_g3,posterior_g4,posterior_phi,posterior_sigma,posterior_beta3,posterior_rho
Absolute Bias,0.103336,0.011946,0.105438,0.001928,0.383312,0.337389,0.221172,0.017070
Standard Error,0.224391,0.092081,0.094180,0.116235,0.091482,0.072797,1.503145,0.210302
95% CI Length,0.867465,0.357093,0.368509,0.458729,0.345781,0.300146,6.207993,0.761855
media,2.614946,0.988054,0.894562,0.998072,0.416688,1.037389,5.221172,0.517070
desviacion,0.224391,0.092081,0.094180,0.116235,0.091482,0.072797,1.503145,0.210302
lower_bound,2.171020,0.808516,0.708419,0.764692,0.238054,0.840968,2.816980,0.204281
upper_bound,3.038485,1.165610,1.076928,1.223420,0.583834,1.141114,9.024973,0.966136
bias,-0.103336,-0.011946,-0.105438,-0.001928,-0.383312,0.337389,0.221172,0.017070
ess,2218.000000,2246.000000,2396.000000,2563.000000,2382.000000,2510.000000,2546.000000,2393.000000
mcse,0.032000,0.005000,0.002000,0.002000,0.002000,0.002000,0.004000,0.002000


0.10& 0.01& 0.11& 0.00& 0.38& 0.34& 0.22& 0.02
0.22& 0.09& 0.09& 0.12& 0.09& 0.07& 1.50& 0.21
0.87& 0.36& 0.37& 0.46& 0.35& 0.30& 6.21& 0.76
2.61& 0.99& 0.89& 1.00& 0.42& 1.04& 5.22& 0.52
0.22& 0.09& 0.09& 0.12& 0.09& 0.07& 1.50& 0.21
2.17& 0.81& 0.71& 0.76& 0.24& 0.84& 2.82& 0.20
3.04& 1.17& 1.08& 1.22& 0.58& 1.14& 9.02& 0.97
-0.10& -0.01& -0.11& -0.00& -0.38& 0.34& 0.22& 0.02
2218.00& 2246.00& 2396.00& 2563.00& 2382.00& 2510.00& 2546.00& 2393.00
0.03& 0.01& 0.00& 0.00& 0.00& 0.00& 0.00& 0.00
25& 25& 27& 28& 26& 28& 28& 27


In [8]:
df=pd.read_csv('Traceplot_simulacion_phi/estimacion_t_simulacion_D8_extremo_V3_V13_c1.csv')
orden = ['posterior_g1', 'posterior_g2', 'posterior_g3',
       'posterior_g4', 'posterior_phi','posterior_sigma','posterior_beta3','posterior_rho']

d=mcmc_summary_statistics(df,[np.exp(1),1,1,1,0.5,1,5,0.5],orden,90,0)
for row in d.T.itertuples():
    if row[0]!='ess_min':
        print('& '.join([f"{x:.2f}" for x in row[1:]]))
    else:
        print('& '.join([f"{x:.0f}" for x in row[1:]]))



d=mcmc_summary_statistics(df,[np.exp(1),1,1,1,0.8,0.7,5,0.5],orden,90,7500)
display(d.T)
for row in d.T.itertuples():
    if row[0]!='ess_min':
        print('& '.join([f"{x:.2f}" for x in row[1:]]))
    else:
        print('& '.join([f"{x:.0f}" for x in row[1:]]))

arviz - WARNING - Shape validation failed: input_shape: (1, 10000), minimum_shape: (chains=2, draws=4)
arviz - WARNING - Shape validation failed: input_shape: (1, 2500), minimum_shape: (chains=2, draws=4)


0.21& 0.03& 0.13& 0.09& 0.20& 0.10& 0.34& 0.20
0.19& 0.08& 0.09& 0.12& 0.07& 0.09& 0.69& 0.16
0.76& 0.32& 0.34& 0.45& 0.30& 0.38& 2.76& 0.59
2.51& 1.03& 0.87& 1.09& 0.30& 1.10& 4.66& 0.70
0.19& 0.08& 0.09& 0.12& 0.07& 0.09& 0.69& 0.16
2.13& 0.87& 0.70& 0.87& 0.17& 0.80& 3.35& 0.39
2.89& 1.19& 1.04& 1.32& 0.47& 1.18& 6.11& 0.98
-0.21& 0.03& -0.13& 0.09& -0.20& 0.10& -0.34& 0.20
9514.00& 9526.00& 10046.00& 9695.00& 9879.00& 10110.00& 10027.00& 9947.00
0.01& 0.00& 0.00& 0.00& 0.00& 0.00& 0.00& 0.00
106& 106& 112& 108& 110& 112& 111& 111


,posterior_g1,posterior_g2,posterior_g3,posterior_g4,posterior_phi,posterior_sigma,posterior_beta3,posterior_rho
Absolute Bias,0.206199,0.029177,0.129977,0.090764,0.500743,0.398023,0.345756,0.198214
Standard Error,0.195862,0.082595,0.086650,0.114739,0.076348,0.091325,0.687598,0.163235
95% CI Length,0.777136,0.315132,0.337697,0.449096,0.295300,0.382150,2.805619,0.581865
media,2.512083,1.029177,0.870023,1.090764,0.299257,1.098023,4.654244,0.698214
desviacion,0.195862,0.082595,0.086650,0.114739,0.076348,0.091325,0.687598,0.163235
lower_bound,2.121367,0.870311,0.706292,0.872009,0.178035,0.799532,3.321485,0.397322
upper_bound,2.898503,1.185444,1.043988,1.321105,0.473335,1.181683,6.127104,0.979187
bias,-0.206199,0.029177,-0.129977,0.090764,-0.500743,0.398023,-0.345756,0.198214
ess,2413.000000,2374.000000,2703.000000,2404.000000,2338.000000,2730.000000,2533.000000,2385.000000
mcse,0.014000,0.004000,0.002000,0.002000,0.002000,0.001000,0.003000,0.002000


0.21& 0.03& 0.13& 0.09& 0.50& 0.40& 0.35& 0.20
0.20& 0.08& 0.09& 0.11& 0.08& 0.09& 0.69& 0.16
0.78& 0.32& 0.34& 0.45& 0.30& 0.38& 2.81& 0.58
2.51& 1.03& 0.87& 1.09& 0.30& 1.10& 4.65& 0.70
0.20& 0.08& 0.09& 0.11& 0.08& 0.09& 0.69& 0.16
2.12& 0.87& 0.71& 0.87& 0.18& 0.80& 3.32& 0.40
2.90& 1.19& 1.04& 1.32& 0.47& 1.18& 6.13& 0.98
-0.21& 0.03& -0.13& 0.09& -0.50& 0.40& -0.35& 0.20
2413.00& 2374.00& 2703.00& 2404.00& 2338.00& 2730.00& 2533.00& 2385.00
0.01& 0.00& 0.00& 0.00& 0.00& 0.00& 0.00& 0.00
27& 26& 30& 27& 26& 30& 28& 26


In [9]:
df=pd.read_csv('Traceplot_simulacion_phi/estimacion_t_simulacion_D8_extremo_V4_V14_c1.csv')
orden = ['posterior_g1', 'posterior_g2', 'posterior_g3',
       'posterior_g4', 'posterior_phi','posterior_sigma','posterior_beta3','posterior_rho']

d=mcmc_summary_statistics(df,[np.exp(1),1,1,1,0.5,1,5,0.5],orden,90,0)
for row in d.T.itertuples():
    if row[0]!='ess_min':
        print('& '.join([f"{x:.2f}" for x in row[1:]]))
    else:
        print('& '.join([f"{x:.0f}" for x in row[1:]]))



d=mcmc_summary_statistics(df,[np.exp(1),1,1,1,0.8,0.7,5,0.5],orden,90,7500)
display(d.T)
for row in d.T.itertuples():
    if row[0]!='ess_min':
        print('& '.join([f"{x:.2f}" for x in row[1:]]))
    else:
        print('& '.join([f"{x:.0f}" for x in row[1:]]))

arviz - WARNING - Shape validation failed: input_shape: (1, 10000), minimum_shape: (chains=2, draws=4)
arviz - WARNING - Shape validation failed: input_shape: (1, 2500), minimum_shape: (chains=2, draws=4)


0.17& 0.01& 0.10& 0.14& 0.09& 0.06& 0.36& 0.02
0.22& 0.08& 0.09& 0.13& 0.09& 0.09& 1.22& 0.20
0.86& 0.33& 0.36& 0.49& 0.38& 0.40& 4.66& 0.72
2.54& 0.99& 0.90& 1.14& 0.41& 1.06& 5.36& 0.48
0.22& 0.08& 0.09& 0.13& 0.09& 0.09& 1.22& 0.20
2.11& 0.82& 0.72& 0.89& 0.27& 0.75& 2.81& 0.21
2.97& 1.15& 1.09& 1.38& 0.66& 1.15& 7.47& 0.93
-0.17& -0.01& -0.10& 0.14& -0.09& 0.06& 0.36& -0.02
9464.00& 9818.00& 10031.00& 9892.00& 9929.00& 9332.00& 9664.00& 9793.00
0.01& 0.00& 0.00& 0.00& 0.00& 0.00& 0.00& 0.00
105& 109& 111& 110& 110& 104& 107& 109


,posterior_g1,posterior_g2,posterior_g3,posterior_g4,posterior_phi,posterior_sigma,posterior_beta3,posterior_rho
Absolute Bias,0.173710,0.013241,0.095233,0.133701,0.387554,0.358824,0.362984,0.022010
Standard Error,0.221161,0.084340,0.090686,0.123073,0.092563,0.090281,1.215750,0.202916
95% CI Length,0.866980,0.332332,0.356210,0.481481,0.390562,0.403358,4.757900,0.727047
media,2.544572,0.986759,0.904767,1.133701,0.412446,1.058824,5.362984,0.477990
desviacion,0.221161,0.084340,0.090686,0.123073,0.092563,0.090281,1.215750,0.202916
lower_bound,2.107785,0.822357,0.727054,0.896878,0.270144,0.744006,2.782465,0.204472
upper_bound,2.974765,1.154689,1.083264,1.378359,0.660706,1.147364,7.540365,0.931519
bias,-0.173710,-0.013241,-0.095233,0.133701,-0.387554,0.358824,0.362984,-0.022010
ess,2389.000000,2584.000000,2506.000000,2477.000000,2737.000000,2462.000000,2408.000000,2520.000000
mcse,0.025000,0.004000,0.002000,0.002000,0.002000,0.002000,0.004000,0.002000


0.17& 0.01& 0.10& 0.13& 0.39& 0.36& 0.36& 0.02
0.22& 0.08& 0.09& 0.12& 0.09& 0.09& 1.22& 0.20
0.87& 0.33& 0.36& 0.48& 0.39& 0.40& 4.76& 0.73
2.54& 0.99& 0.90& 1.13& 0.41& 1.06& 5.36& 0.48
0.22& 0.08& 0.09& 0.12& 0.09& 0.09& 1.22& 0.20
2.11& 0.82& 0.73& 0.90& 0.27& 0.74& 2.78& 0.20
2.97& 1.15& 1.08& 1.38& 0.66& 1.15& 7.54& 0.93
-0.17& -0.01& -0.10& 0.13& -0.39& 0.36& 0.36& -0.02
2389.00& 2584.00& 2506.00& 2477.00& 2737.00& 2462.00& 2408.00& 2520.00
0.03& 0.00& 0.00& 0.00& 0.00& 0.00& 0.00& 0.00
27& 29& 28& 28& 30& 27& 27& 28


In [10]:
df=pd.read_csv('Traceplot_simulacion_phi/estimacion_t_simulacion_D8_extremo_V6_V16_c1.csv')
orden = ['posterior_g1', 'posterior_g2', 'posterior_g3',
       'posterior_g4', 'posterior_phi','posterior_sigma','posterior_beta3','posterior_rho']

d=mcmc_summary_statistics(df,[np.exp(1),1,1,1,0.5,1,5,0.5],orden,90,0)
for row in d.T.itertuples():
    if row[0]!='ess_min':
        print('& '.join([f"{x:.2f}" for x in row[1:]]))
    else:
        print('& '.join([f"{x:.0f}" for x in row[1:]]))



d=mcmc_summary_statistics(df,[np.exp(1),1,1,1,0.8,0.7,5,0.5],orden,90,7500)
display(d.T)
for row in d.T.itertuples():
    if row[0]!='ess_min':
        print('& '.join([f"{x:.2f}" for x in row[1:]]))
    else:
        print('& '.join([f"{x:.0f}" for x in row[1:]]))

arviz - WARNING - Shape validation failed: input_shape: (1, 10000), minimum_shape: (chains=2, draws=4)
arviz - WARNING - Shape validation failed: input_shape: (1, 2500), minimum_shape: (chains=2, draws=4)


0.22& 0.04& 0.09& 0.02& 0.02& 0.01& 0.25& 0.04
0.22& 0.09& 0.08& 0.12& 0.09& 0.12& 0.67& 0.19
0.89& 0.35& 0.31& 0.48& 0.37& 0.47& 2.79& 0.68
2.49& 1.04& 0.91& 1.02& 0.52& 0.99& 4.75& 0.54
0.22& 0.09& 0.08& 0.12& 0.09& 0.12& 0.67& 0.19
2.04& 0.87& 0.76& 0.78& 0.36& 0.65& 3.19& 0.27
2.93& 1.22& 1.07& 1.26& 0.73& 1.12& 5.98& 0.95
-0.22& 0.04& -0.09& 0.02& 0.02& -0.01& -0.25& 0.04
9261.00& 9190.00& 9786.00& 9895.00& 9872.00& 10214.00& 9062.00& 10053.00
0.01& 0.00& 0.00& 0.00& 0.00& 0.00& 0.00& 0.00
103& 102& 109& 110& 110& 113& 101& 112


,posterior_g1,posterior_g2,posterior_g3,posterior_g4,posterior_phi,posterior_sigma,posterior_beta3,posterior_rho
Absolute Bias,0.225399,0.042634,0.089302,0.023206,0.283498,0.289476,0.232365,0.041813
Standard Error,0.226880,0.088973,0.079471,0.122217,0.095914,0.122989,0.677735,0.188315
95% CI Length,0.896446,0.355490,0.314355,0.477955,0.372241,0.476700,2.768930,0.691983
media,2.492883,1.042634,0.910698,1.023206,0.516502,0.989476,4.767635,0.541813
desviacion,0.226880,0.088973,0.079471,0.122217,0.095914,0.122989,0.677735,0.188315
lower_bound,2.031540,0.869023,0.756246,0.785178,0.362454,0.649433,3.253813,0.260534
upper_bound,2.927986,1.224513,1.070601,1.263134,0.734695,1.126132,6.022743,0.952518
bias,-0.225399,0.042634,-0.089302,0.023206,-0.283498,0.289476,-0.232365,0.041813
ess,2276.000000,2168.000000,2447.000000,2541.000000,2536.000000,2311.000000,2161.000000,2403.000000
mcse,0.014000,0.005000,0.002000,0.002000,0.002000,0.002000,0.004000,0.002000


0.23& 0.04& 0.09& 0.02& 0.28& 0.29& 0.23& 0.04
0.23& 0.09& 0.08& 0.12& 0.10& 0.12& 0.68& 0.19
0.90& 0.36& 0.31& 0.48& 0.37& 0.48& 2.77& 0.69
2.49& 1.04& 0.91& 1.02& 0.52& 0.99& 4.77& 0.54
0.23& 0.09& 0.08& 0.12& 0.10& 0.12& 0.68& 0.19
2.03& 0.87& 0.76& 0.79& 0.36& 0.65& 3.25& 0.26
2.93& 1.22& 1.07& 1.26& 0.73& 1.13& 6.02& 0.95
-0.23& 0.04& -0.09& 0.02& -0.28& 0.29& -0.23& 0.04
2276.00& 2168.00& 2447.00& 2541.00& 2536.00& 2311.00& 2161.00& 2403.00
0.01& 0.01& 0.00& 0.00& 0.00& 0.00& 0.00& 0.00
25& 24& 27& 28& 28& 26& 24& 27


# Yadav

In [11]:
df=pd.read_csv('traceplot_simulacion_phi/estimacion_t_simulacion_DY_extremo_V1_V112_15_c1.csv')
orden = ['posterior_g1', 'posterior_g2', 'posterior_g3',
       'posterior_g4', 'posterior_beta1','posterior_beta2','posterior_beta3','posterior_rho']

d=mcmc_summary_statistics(df,[np.exp(1),1,1,1,0.8,0.7,5,0.5],orden,60+27,0)
for row in d.T.itertuples():
    if row[0]!='ess_min':
        print('& '.join([f"{x:.2f}" for x in row[1:]]))
    else:
        print('& '.join([f"{x:.0f}" for x in row[1:]]))



d=mcmc_summary_statistics(df,[np.exp(1),1,1,1,0.8,0.7,5,0.5],orden,60+27,7500)
display(d.T)
for row in d.T.itertuples():
    if row[0]!='ess_min':
        print('& '.join([f"{x:.2f}" for x in row[1:]]))
    else:
        print('& '.join([f"{x:.0f}" for x in row[1:]]))

arviz - WARNING - Shape validation failed: input_shape: (1, 10000), minimum_shape: (chains=2, draws=4)
arviz - WARNING - Shape validation failed: input_shape: (1, 2500), minimum_shape: (chains=2, draws=4)


0.09& 0.00& 0.00& 0.02& 0.01& 0.02& 0.16& 0.15
0.27& 0.08& 0.09& 0.09& 0.06& 0.10& 1.28& 0.21
1.07& 0.33& 0.34& 0.36& 0.09& 0.41& 4.92& 0.75
2.81& 1.00& 1.00& 0.98& 0.81& 0.68& 4.84& 0.65
0.27& 0.08& 0.09& 0.09& 0.06& 0.10& 1.28& 0.21
2.27& 0.83& 0.83& 0.80& 0.76& 0.48& 2.85& 0.23
3.34& 1.16& 1.17& 1.16& 0.85& 0.90& 7.77& 0.98
0.09& -0.00& -0.00& -0.02& 0.01& -0.02& -0.16& 0.15
9926.00& 9780.00& 9645.00& 9812.00& 9591.00& 9987.00& 9678.00& 9876.00
0.00& 0.00& 0.01& 0.00& 0.00& 0.00& 0.00& 0.00
114& 112& 111& 113& 110& 115& 111& 114


,posterior_g1,posterior_g2,posterior_g3,posterior_g4,posterior_beta1,posterior_beta2,posterior_beta3,posterior_rho
Absolute Bias,0.095627,0.007252,0.001407,0.013549,0.008843,0.023972,0.194828,0.149625
Standard Error,0.276174,0.083646,0.087058,0.088846,0.058499,0.098898,1.266713,0.211084
95% CI Length,1.065496,0.328798,0.332766,0.353367,0.088364,0.410317,4.863088,0.740325
media,2.813909,0.992748,1.001407,0.986451,0.808843,0.676028,4.805172,0.649625
desviacion,0.276174,0.083646,0.087058,0.088846,0.058499,0.098898,1.266713,0.211084
lower_bound,2.276433,0.831769,0.833321,0.802998,0.758655,0.489341,2.852536,0.237697
upper_bound,3.341930,1.160567,1.166087,1.156364,0.847019,0.899658,7.715624,0.978023
bias,0.095627,-0.007252,0.001407,-0.013549,0.008843,-0.023972,-0.194828,0.149625
ess,2462.000000,2625.000000,2498.000000,2398.000000,2274.000000,2400.000000,2573.000000,2758.000000
mcse,0.001000,0.002000,0.025000,0.006000,0.002000,0.002000,0.002000,0.004000


0.10& 0.01& 0.00& 0.01& 0.01& 0.02& 0.19& 0.15
0.28& 0.08& 0.09& 0.09& 0.06& 0.10& 1.27& 0.21
1.07& 0.33& 0.33& 0.35& 0.09& 0.41& 4.86& 0.74
2.81& 0.99& 1.00& 0.99& 0.81& 0.68& 4.81& 0.65
0.28& 0.08& 0.09& 0.09& 0.06& 0.10& 1.27& 0.21
2.28& 0.83& 0.83& 0.80& 0.76& 0.49& 2.85& 0.24
3.34& 1.16& 1.17& 1.16& 0.85& 0.90& 7.72& 0.98
0.10& -0.01& 0.00& -0.01& 0.01& -0.02& -0.19& 0.15
2462.00& 2625.00& 2498.00& 2398.00& 2274.00& 2400.00& 2573.00& 2758.00
0.00& 0.00& 0.03& 0.01& 0.00& 0.00& 0.00& 0.00
28& 30& 29& 28& 26& 28& 30& 32


In [12]:
df=pd.read_csv('traceplot_simulacion_phi/estimacion_t_simulacion_DY_extremo_V2_V122_15_c1.csv')
orden = ['posterior_g1', 'posterior_g2', 'posterior_g3',
       'posterior_g4', 'posterior_beta1','posterior_beta2','posterior_beta3','posterior_rho']

d=mcmc_summary_statistics(df,[np.exp(1),1,1,1,0.8,0.7,5,0.5],orden,60+27,0)
for row in d.T.itertuples():
    if row[0]!='ess_min':
        print('& '.join([f"{x:.2f}" for x in row[1:]]))
    else:
        print('& '.join([f"{x:.0f}" for x in row[1:]]))



d=mcmc_summary_statistics(df,[np.exp(1),1,1,1,0.8,0.7,5,0.5],orden,60+27,7500)
display(d.T)
for row in d.T.itertuples():
    if row[0]!='ess_min':
        print('& '.join([f"{x:.2f}" for x in row[1:]]))
    else:
        print('& '.join([f"{x:.0f}" for x in row[1:]]))

arviz - WARNING - Shape validation failed: input_shape: (1, 10000), minimum_shape: (chains=2, draws=4)
arviz - WARNING - Shape validation failed: input_shape: (1, 2500), minimum_shape: (chains=2, draws=4)


0.12& 0.06& 0.01& 0.11& 0.01& 0.00& 0.20& 0.12
0.28& 0.09& 0.08& 0.10& 0.02& 0.11& 1.40& 0.22
1.10& 0.35& 0.33& 0.38& 0.09& 0.46& 5.14& 0.73
2.83& 1.06& 0.99& 1.11& 0.81& 0.70& 4.80& 0.62
0.28& 0.09& 0.08& 0.10& 0.02& 0.11& 1.40& 0.22
2.29& 0.89& 0.83& 0.92& 0.78& 0.43& 2.66& 0.25
3.39& 1.24& 1.16& 1.30& 0.86& 0.89& 7.80& 0.98
0.12& 0.06& -0.01& 0.11& 0.01& 0.00& -0.20& 0.12
9955.00& 9703.00& 9294.00& 9524.00& 10085.00& 9489.00& 10261.00& 9640.00
0.00& 0.00& 0.01& 0.00& 0.00& 0.00& 0.00& 0.00
114& 112& 107& 109& 116& 109& 118& 111


,posterior_g1,posterior_g2,posterior_g3,posterior_g4,posterior_beta1,posterior_beta2,posterior_beta3,posterior_rho
Absolute Bias,0.115133,0.064791,0.012453,0.106333,0.011379,0.003112,0.206946,0.118985
Standard Error,0.283461,0.089675,0.082659,0.094794,0.020401,0.110368,1.356293,0.215582
95% CI Length,1.116401,0.348312,0.325444,0.375429,0.086055,0.490175,5.062819,0.731239
media,2.833414,1.064791,0.987547,1.106333,0.811379,0.703112,4.793054,0.618985
desviacion,0.283461,0.089675,0.082659,0.094794,0.020401,0.110368,1.356293,0.215582
lower_bound,2.288683,0.888214,0.828535,0.917253,0.779221,0.397992,2.677652,0.249982
upper_bound,3.405083,1.236527,1.153979,1.292682,0.865276,0.888167,7.740471,0.981222
bias,0.115133,0.064791,-0.012453,0.106333,0.011379,0.003112,-0.206946,0.118985
ess,2443.000000,2432.000000,2436.000000,2322.000000,2436.000000,2337.000000,2507.000000,2405.000000
mcse,0.000000,0.002000,0.027000,0.006000,0.002000,0.002000,0.002000,0.004000


0.12& 0.06& 0.01& 0.11& 0.01& 0.00& 0.21& 0.12
0.28& 0.09& 0.08& 0.09& 0.02& 0.11& 1.36& 0.22
1.12& 0.35& 0.33& 0.38& 0.09& 0.49& 5.06& 0.73
2.83& 1.06& 0.99& 1.11& 0.81& 0.70& 4.79& 0.62
0.28& 0.09& 0.08& 0.09& 0.02& 0.11& 1.36& 0.22
2.29& 0.89& 0.83& 0.92& 0.78& 0.40& 2.68& 0.25
3.41& 1.24& 1.15& 1.29& 0.87& 0.89& 7.74& 0.98
0.12& 0.06& -0.01& 0.11& 0.01& 0.00& -0.21& 0.12
2443.00& 2432.00& 2436.00& 2322.00& 2436.00& 2337.00& 2507.00& 2405.00
0.00& 0.00& 0.03& 0.01& 0.00& 0.00& 0.00& 0.00
28& 28& 28& 27& 28& 27& 29& 28


In [13]:
df=pd.read_csv('traceplot_simulacion_phi/estimacion_t_simulacion_DY_extremo_V3_V132_15_c1.csv')
orden = ['posterior_g1', 'posterior_g2', 'posterior_g3',
       'posterior_g4', 'posterior_beta1','posterior_beta2','posterior_beta3','posterior_rho']

d=mcmc_summary_statistics(df,[np.exp(1),1,1,1,0.8,0.7,5,0.5],orden,60+30,0)
for row in d.T.itertuples():
    if row[0]!='ess_min':
        print('& '.join([f"{x:.2f}" for x in row[1:]]))
    else:
        print('& '.join([f"{x:.0f}" for x in row[1:]]))



d=mcmc_summary_statistics(df,[np.exp(1),1,1,1,0.8,0.7,5,0.5],orden,60+30,7500)
display(d.T)
for row in d.T.itertuples():
    if row[0]!='ess_min':
        print('& '.join([f"{x:.2f}" for x in row[1:]]))
    else:
        print('& '.join([f"{x:.0f}" for x in row[1:]]))

arviz - WARNING - Shape validation failed: input_shape: (1, 10000), minimum_shape: (chains=2, draws=4)
arviz - WARNING - Shape validation failed: input_shape: (1, 2500), minimum_shape: (chains=2, draws=4)


0.03& 0.02& 0.04& 0.12& 0.00& 0.03& 1.10& 0.15
0.22& 0.07& 0.08& 0.08& 0.02& 0.08& 1.44& 0.21
0.87& 0.29& 0.33& 0.33& 0.07& 0.33& 5.52& 0.79
2.69& 1.02& 1.04& 1.12& 0.80& 0.73& 6.10& 0.35
0.22& 0.07& 0.08& 0.08& 0.02& 0.08& 1.44& 0.21
2.25& 0.88& 0.87& 0.95& 0.77& 0.61& 2.94& 0.15
3.12& 1.17& 1.20& 1.28& 0.84& 0.94& 8.45& 0.95
-0.03& 0.02& 0.04& 0.12& -0.00& 0.03& 1.10& -0.15
10490.00& 9729.00& 9608.00& 9396.00& 9710.00& 9152.00& 9308.00& 9353.00
0.00& 0.00& 0.01& 0.00& 0.00& 0.00& 0.00& 0.00
117& 108& 107& 104& 108& 102& 103& 104


,posterior_g1,posterior_g2,posterior_g3,posterior_g4,posterior_beta1,posterior_beta2,posterior_beta3,posterior_rho
Absolute Bias,0.027432,0.021838,0.039235,0.120339,0.003631,0.027638,1.119583,0.148970
Standard Error,0.220185,0.074561,0.083957,0.084094,0.018265,0.075917,1.464429,0.208707
95% CI Length,0.869755,0.288806,0.321645,0.325420,0.074828,0.320863,5.519913,0.790609
media,2.690849,1.021838,1.039235,1.120339,0.796369,0.727638,6.119583,0.351030
desviacion,0.220185,0.074561,0.083957,0.084094,0.018265,0.075917,1.464429,0.208707
lower_bound,2.262618,0.874156,0.882369,0.956366,0.765243,0.609466,2.928941,0.151838
upper_bound,3.132374,1.162962,1.204014,1.281786,0.840071,0.930329,8.448853,0.942447
bias,-0.027432,0.021838,0.039235,0.120339,-0.003631,0.027638,1.119583,-0.148970
ess,2352.000000,2210.000000,2195.000000,2301.000000,2213.000000,2173.000000,2400.000000,2248.000000
mcse,0.000000,0.002000,0.031000,0.005000,0.002000,0.002000,0.002000,0.004000


0.03& 0.02& 0.04& 0.12& 0.00& 0.03& 1.12& 0.15
0.22& 0.07& 0.08& 0.08& 0.02& 0.08& 1.46& 0.21
0.87& 0.29& 0.32& 0.33& 0.07& 0.32& 5.52& 0.79
2.69& 1.02& 1.04& 1.12& 0.80& 0.73& 6.12& 0.35
0.22& 0.07& 0.08& 0.08& 0.02& 0.08& 1.46& 0.21
2.26& 0.87& 0.88& 0.96& 0.77& 0.61& 2.93& 0.15
3.13& 1.16& 1.20& 1.28& 0.84& 0.93& 8.45& 0.94
-0.03& 0.02& 0.04& 0.12& -0.00& 0.03& 1.12& -0.15
2352.00& 2210.00& 2195.00& 2301.00& 2213.00& 2173.00& 2400.00& 2248.00
0.00& 0.00& 0.03& 0.01& 0.00& 0.00& 0.00& 0.00
26& 25& 24& 26& 25& 24& 27& 25


In [14]:
df=pd.read_csv('traceplot_simulacion_phi/estimacion_t_simulacion_DY_extremo_V4_V142_15_c1.csv')
orden = ['posterior_g1', 'posterior_g2', 'posterior_g3',
       'posterior_g4', 'posterior_beta1','posterior_beta2','posterior_beta3','posterior_rho']

d=mcmc_summary_statistics(df,[np.exp(1),1,1,1,0.8,0.7,5,0.5],orden,60+30,0)
for row in d.T.itertuples():
    if row[0]!='ess_min':
        print('& '.join([f"{x:.2f}" for x in row[1:]]))
    else:
        print('& '.join([f"{x:.0f}" for x in row[1:]]))



d=mcmc_summary_statistics(df,[np.exp(1),1,1,1,0.8,0.7,5,0.5],orden,(14*60+17)/4,7500)
display(d.T)
for row in d.T.itertuples():
    if row[0]!='ess_min':
        print('& '.join([f"{x:.2f}" for x in row[1:]]))
    else:
        print('& '.join([f"{x:.0f}" for x in row[1:]]))

arviz - WARNING - Shape validation failed: input_shape: (1, 10000), minimum_shape: (chains=2, draws=4)
arviz - WARNING - Shape validation failed: input_shape: (1, 2500), minimum_shape: (chains=2, draws=4)


0.05& 0.01& 0.02& 0.01& 0.01& 0.01& 0.59& 0.13
0.22& 0.08& 0.07& 0.09& 0.01& 0.06& 1.13& 0.17
0.85& 0.30& 0.27& 0.34& 0.06& 0.28& 4.45& 0.68
2.77& 1.01& 0.98& 0.99& 0.79& 0.71& 5.59& 0.37
0.22& 0.08& 0.07& 0.09& 0.01& 0.06& 1.13& 0.17
2.35& 0.87& 0.85& 0.83& 0.77& 0.61& 2.89& 0.19
3.20& 1.17& 1.11& 1.16& 0.83& 0.90& 7.34& 0.86
0.05& 0.01& -0.02& -0.01& -0.01& 0.01& 0.59& -0.13
9766.00& 9656.00& 8586.00& 9209.00& 8888.00& 10233.00& 9525.00& 9692.00
0.00& 0.00& 0.01& 0.00& 0.00& 0.00& 0.00& 0.00
109& 107& 95& 102& 99& 114& 106& 108


,posterior_g1,posterior_g2,posterior_g3,posterior_g4,posterior_beta1,posterior_beta2,posterior_beta3,posterior_rho
Absolute Bias,0.055258,0.015210,0.020623,0.003972,0.008435,0.012115,0.568112,0.131111
Standard Error,0.220630,0.078451,0.068034,0.086143,0.014613,0.059924,1.115987,0.169532
95% CI Length,0.858744,0.304022,0.261420,0.337054,0.060993,0.271594,4.376326,0.685738
media,2.773540,1.015210,0.979377,0.996028,0.791565,0.712115,5.568112,0.368889
desviacion,0.220630,0.078451,0.068034,0.086143,0.014613,0.059924,1.115987,0.169532
lower_bound,2.345524,0.865722,0.850712,0.825572,0.766427,0.614903,2.859310,0.187807
upper_bound,3.204269,1.169743,1.112132,1.162626,0.827421,0.886497,7.235636,0.873545
bias,0.055258,0.015210,-0.020623,-0.003972,-0.008435,0.012115,0.568112,-0.131111
ess,2530.000000,2389.000000,1979.000000,2313.000000,1885.000000,2552.000000,2416.000000,2393.000000
mcse,0.000000,0.001000,0.024000,0.005000,0.002000,0.001000,0.002000,0.003000


0.06& 0.02& 0.02& 0.00& 0.01& 0.01& 0.57& 0.13
0.22& 0.08& 0.07& 0.09& 0.01& 0.06& 1.12& 0.17
0.86& 0.30& 0.26& 0.34& 0.06& 0.27& 4.38& 0.69
2.77& 1.02& 0.98& 1.00& 0.79& 0.71& 5.57& 0.37
0.22& 0.08& 0.07& 0.09& 0.01& 0.06& 1.12& 0.17
2.35& 0.87& 0.85& 0.83& 0.77& 0.61& 2.86& 0.19
3.20& 1.17& 1.11& 1.16& 0.83& 0.89& 7.24& 0.87
0.06& 0.02& -0.02& -0.00& -0.01& 0.01& 0.57& -0.13
2530.00& 2389.00& 1979.00& 2313.00& 1885.00& 2552.00& 2416.00& 2393.00
0.00& 0.00& 0.02& 0.01& 0.00& 0.00& 0.00& 0.00
12& 11& 9& 11& 9& 12& 11& 11


In [15]:
df=pd.read_csv('traceplot_simulacion_phi/estimacion_t_simulacion_DY_extremo_V6_V162_15_c1.csv')
orden = ['posterior_g1', 'posterior_g2', 'posterior_g3',
       'posterior_g4', 'posterior_beta1','posterior_beta2','posterior_beta3','posterior_rho']

d=mcmc_summary_statistics(df,[np.exp(1),1,1,1,0.8,0.7,5,0.5],orden,60+31,0)
for row in d.T.itertuples():
    if row[0]!='ess_min':
        print('& '.join([f"{x:.2f}" for x in row[1:]]))
    else:
        print('& '.join([f"{x:.0f}" for x in row[1:]]))



d=mcmc_summary_statistics(df,[np.exp(1),1,1,1,0.8,0.7,5,0.5],orden,(60+31)/4,7500)
display(d.T)
for row in d.T.itertuples():
    if row[0]!='ess_min':
        print('& '.join([f"{x:.2f}" for x in row[1:]]))
    else:
        print('& '.join([f"{x:.0f}" for x in row[1:]]))

arviz - WARNING - Shape validation failed: input_shape: (1, 10000), minimum_shape: (chains=2, draws=4)
arviz - WARNING - Shape validation failed: input_shape: (1, 2500), minimum_shape: (chains=2, draws=4)


0.08& 0.07& 0.04& 0.03& 0.00& 0.01& 0.28& 0.04
0.22& 0.08& 0.08& 0.08& 0.03& 0.09& 1.24& 0.20
0.87& 0.29& 0.31& 0.31& 0.11& 0.35& 4.88& 0.72
2.64& 0.93& 0.96& 0.97& 0.80& 0.71& 5.28& 0.54
0.22& 0.08& 0.08& 0.08& 0.03& 0.09& 1.24& 0.20
2.21& 0.78& 0.81& 0.82& 0.74& 0.53& 3.17& 0.22
3.08& 1.08& 1.12& 1.13& 0.85& 0.88& 8.05& 0.94
-0.08& -0.07& -0.04& -0.03& -0.00& 0.01& 0.28& 0.04
9905.00& 10137.00& 9682.00& 10116.00& 9445.00& 10009.00& 9491.00& 9641.00
0.00& 0.00& 0.01& 0.00& 0.00& 0.00& 0.00& 0.00
109& 111& 106& 111& 104& 110& 104& 106


,posterior_g1,posterior_g2,posterior_g3,posterior_g4,posterior_beta1,posterior_beta2,posterior_beta3,posterior_rho
Absolute Bias,0.069232,0.068285,0.041589,0.026912,0.002052,0.010586,0.309146,0.036018
Standard Error,0.216769,0.075848,0.080960,0.081907,0.030019,0.089280,1.261364,0.196008
95% CI Length,0.856318,0.301266,0.316668,0.321894,0.115851,0.360042,4.929244,0.721892
media,2.649050,0.931715,0.958411,0.973088,0.797948,0.710586,5.309146,0.536018
desviacion,0.216769,0.075848,0.080960,0.081907,0.030019,0.089280,1.261364,0.196008
lower_bound,2.215111,0.781289,0.801346,0.811628,0.737176,0.527951,3.263610,0.217175
upper_bound,3.071428,1.082555,1.118015,1.133522,0.853027,0.887993,8.192854,0.939067
bias,-0.069232,-0.068285,-0.041589,-0.026912,-0.002052,0.010586,0.309146,0.036018
ess,2370.000000,2468.000000,2333.000000,2472.000000,2306.000000,2476.000000,2330.000000,2152.000000
mcse,0.001000,0.002000,0.026000,0.004000,0.002000,0.002000,0.002000,0.004000


0.07& 0.07& 0.04& 0.03& 0.00& 0.01& 0.31& 0.04
0.22& 0.08& 0.08& 0.08& 0.03& 0.09& 1.26& 0.20
0.86& 0.30& 0.32& 0.32& 0.12& 0.36& 4.93& 0.72
2.65& 0.93& 0.96& 0.97& 0.80& 0.71& 5.31& 0.54
0.22& 0.08& 0.08& 0.08& 0.03& 0.09& 1.26& 0.20
2.22& 0.78& 0.80& 0.81& 0.74& 0.53& 3.26& 0.22
3.07& 1.08& 1.12& 1.13& 0.85& 0.89& 8.19& 0.94
-0.07& -0.07& -0.04& -0.03& -0.00& 0.01& 0.31& 0.04
2370.00& 2468.00& 2333.00& 2472.00& 2306.00& 2476.00& 2330.00& 2152.00
0.00& 0.00& 0.03& 0.00& 0.00& 0.00& 0.00& 0.00
104& 108& 103& 109& 101& 109& 102& 95


## D8